# 13. 최종모형 v2

`12_방법론_감사`(`../방법론_감사.md`)의 '수정 필요' 항목을 반영한 최종모형이다. 문서 정리본은 `../최종모형_v2.md`.

| 바뀐 것 | v1 (`10_최종모형`) | v2 |
|---|---|---|
| 핵심 변수 | 보행 20분 이내 / 20~60분 더미 | **log(보행시간)** 연속형 |
| 통제 변수 | IC, 경쟁 목적지, 20~50대 인구, 건축물 | **IC 도로거리** |
| offset 가정 | 보고 안 함 | **θ=1 검정** + θ 자유 모형 + 독립 규모 통제 모형 |
| pseudo R² | 0.42 | **R²_D 분해** (전체 / 노선 FE만 / 변수 추가분) |
| 구간 더미 | 본 결과 | 보조 결과 (해석용) |

유지: 셀 × 노선, 평일 07:00~09:59, Poisson QMLE, offset log(2024 승차), 노선 고정효과, 셀 군집 SE.

## 식
$$
\log E[Y^{2025}_{ir}] = \log Y^{2024}_{ir} + \beta_1 \log(\text{보행시간}_i) + \beta_2 \, \text{IC도로거리}_i + \gamma_r
$$
- 보행시간이 2배가 될 때 변화율 차이 = $2^{\beta_1} - 1$
- IC가 1km 멀어질 때 변화율 차이 = $e^{\beta_2} - 1$

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

import statsmodels.api as sm
from scipy.stats import norm
from statsmodels.stats.outliers_influence import variance_inflation_factor

data = load_model_data()
data["log_walk"] = np.log(data["walk_min"])
X_V2 = ["log_walk", "ic_access"]
trips = load_trips().dropna(subset=["셀 ID"])


def build_pairs(hours):
    frame = (trips[trips["시"].between(*hours)]
             .pivot_table(index=["셀 ID", "노선"], columns="연도", values="승차인원", aggfunc="sum", fill_value=0)
             .reset_index().rename(columns={2024: "y2024", 2025: "y2025"}))
    frame["셀 ID"] = frame["셀 ID"].astype(int)
    frame = frame.merge(data[["셀 ID", "walk_min", "log_walk", "walk_20", "walk_20_60", "ic_access"]], on="셀 ID")
    return frame[frame.y2024 > 0].reset_index(drop=True)


morning, allday = build_pairs(MORNING_HOURS), build_pairs(ALL_DAY_HOURS)

# 2024년 오전 외 시간대 승차 (오전 잡음과 독립적인 규모 측정치)
other = (trips[~trips["시"].between(*MORNING_HOURS)]
         .pivot_table(index=["셀 ID", "노선"], columns="연도", values="승차인원", aggfunc="sum", fill_value=0)
         .reset_index().rename(columns={2024: "other2024"}))
other["셀 ID"] = other["셀 ID"].astype(int)
morning = morning.merge(other[["셀 ID", "노선", "other2024"]], on=["셀 ID", "노선"], how="left").fillna({"other2024": 0})
for frame in (morning, allday):
    frame["log_y2024"] = np.log(frame.y2024)
morning["log_other2024"] = np.log1p(morning.other2024)

for name, frame in [("오전(본)", morning), ("종일", allday)]:
    print(f"{name}: 셀×노선 {len(frame)}쌍, 셀 {frame['셀 ID'].nunique()}개, 노선 {frame.노선.nunique()}개")


def fit(frame, x_cols, offset=True, extra=None, drop_routes=()):
    frame = frame[~frame.노선.isin(drop_routes)].reset_index(drop=True)
    X = pd.concat([frame[x_cols + ([extra] if extra else [])],
                   pd.get_dummies(frame.노선, prefix="노선", drop_first=True, dtype=float)], axis=1)
    model = sm.GLM(frame.y2025, sm.add_constant(X), family=sm.families.Poisson(),
                   offset=np.log(frame.y2024) if offset else None)
    return model.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(frame["셀 ID"])[0]})


def report(result):
    """보행시간 2배당, IC 1km당 변화율 차이와 95% CI."""
    ci = result.conf_int()
    rows = {}
    for name, label, transform in [("log_walk", "보행시간 2배당", lambda b: (2 ** b - 1) * 100),
                                   ("ic_access", "IC 도로거리 1km당", lambda b: (np.exp(b) - 1) * 100)]:
        if name in result.params:
            rows[label] = {"β": result.params[name], "변화율 차이(%)": transform(result.params[name]),
                           "95% CI 하한(%)": transform(ci.loc[name, 0]), "95% CI 상한(%)": transform(ci.loc[name, 1]),
                           "p": result.pvalues[name]}
    return pd.DataFrame(rows).T

오전(본): 셀×노선 186쌍, 셀 118개, 노선 24개
종일: 셀×노선 220쌍, 셀 132개, 노선 24개


## 1. 본 결과 (오전 07~09시)

In [2]:
main = fit(morning, X_V2)
display(report(main).round(4))

b = main.params["log_walk"]
print("해석용: 보행시간별 변화율 비교 (60분 셀 대비)")
pd.DataFrame({"보행시간(분)": [10, 20, 30, 60, 120],
              "60분 셀 대비 변화율 차이(%)": [((m / 60) ** b - 1) * 100 for m in (10, 20, 30, 60, 120)]}).round(1)

,β,변화율 차이(%),95% CI 하한(%),95% CI 상한(%),p
보행시간 2배당,0.3462,27.1165,13.9093,41.8550,0.0000
IC 도로거리 1km당,-0.1236,-11.6237,-17.7447,-5.0472,0.0007


해석용: 보행시간별 변화율 비교 (60분 셀 대비)


,보행시간(분),60분 셀 대비 변화율 차이(%)
0,10,-46.2
1,20,-31.6
2,30,-21.3
3,60,0.0
4,120,27.1


## 2. offset 가정 검정 (θ = 1)

In [3]:
free = fit(morning, X_V2, offset=False, extra="log_y2024")
theta, se = free.params["log_y2024"], free.bse["log_y2024"]
z = (theta - 1) / se
print(f"θ = {theta:.3f} (SE {se:.3f}), H0: θ=1 → z = {z:.2f}, p = {2 * norm.sf(abs(z)):.4f}")

indep = fit(morning, X_V2, extra="log_other2024")
print(f"독립 규모 통제 log(1+2024 오전 외 승차)의 계수 = {indep.params['log_other2024']:.3f} (p = {indep.pvalues['log_other2024']:.3f})")

pd.concat({"본 결과 (θ=1)": report(main), "θ 자유 추정": report(free), "θ=1 + 독립 규모 통제": report(indep)}, axis=0).round(4)

θ = 0.852 (SE 0.071), H0: θ=1 → z = -2.07, p = 0.0388
독립 규모 통제 log(1+2024 오전 외 승차)의 계수 = 0.016 (p = 0.806)


β  변화율 차이(%)  95% CI 하한(%)  95% CI 상한(%)       p
본 결과 (θ=1)     보행시간 2배당      0.3462    27.1165       13.9093       41.8550  0.0000
               IC 도로거리 1km당 -0.1236   -11.6237      -17.7447       -5.0472  0.0007
θ 자유 추정        보행시간 2배당      0.2295    17.2422        3.3594       32.9897  0.0134
               IC 도로거리 1km당 -0.1059   -10.0444      -15.8815       -3.8023  0.0020
θ=1 + 독립 규모 통제 보행시간 2배당      0.3574    28.1108       15.0892       42.6058  0.0000
               IC 도로거리 1km당 -0.1238   -11.6413      -17.7830       -5.0409  0.0008

→ θ<1은 하루치 오전 기준값의 측정 잡음(평균회귀)으로 판단된다. 오전 잡음과 독립적인 규모 측정치(같은 쌍의 2024년 오전 외 승차)로 통제하면, 그 계수는 0에 가깝고 보행시간 효과는 유지된다.

## 3. pseudo R² (deviance 기반) 분해

In [4]:
def deviance(frame, x_cols, route_fe):
    parts = [frame[x_cols]] if x_cols else []
    if route_fe:
        parts.append(pd.get_dummies(frame.노선, prefix="노선", drop_first=True, dtype=float))
    X = sm.add_constant(pd.concat(parts, axis=1)) if parts else np.ones((len(frame), 1))
    return sm.GLM(frame.y2025, X, family=sm.families.Poisson(), offset=np.log(frame.y2024)).fit().deviance


rows = []
for name, frame in [("오전(본)", morning), ("종일", allday)]:
    d_null, d_route = deviance(frame, [], False), deviance(frame, [], True)
    d_x, d_full = deviance(frame, X_V2, False), deviance(frame, X_V2, True)
    rows.append({"시간대": name, "R²_D 전체": 1 - d_full / d_null, "R²_D 노선 FE만": 1 - d_route / d_null,
                 "R²_D 변수만 (FE 없음)": 1 - d_x / d_null, "부분 R²_D (노선 FE 위에 변수 추가분)": 1 - d_full / d_route})
pd.DataFrame(rows).round(3)

,시간대,R²_D 전체,R²_D 노선 FE만,R²_D 변수만 (FE 없음),부분 R²_D (노선 FE 위에 변수 추가분)
0,오전(본),0.436,0.346,0.070,0.137
1,종일,0.401,0.296,0.035,0.149


R²_D = 1 − D(모형) / D(상수 + offset). D는 Poisson deviance다. null 모형에 offset이 있으므로 '모든 쌍이 같은 변화율'이라는 모형 대비 설명분이다. OLS R²와 비교하지 않는다.

## 4. 노선 1100 준완전분리 점검

In [5]:
by_route = morning.groupby("노선")[["y2024", "y2025"]].sum()
separated = by_route[by_route.y2025 == 0].index.tolist()
print("2025년 오전 승차가 모두 0인 노선:", separated, "| 해당 쌍:", int(morning.노선.isin(separated).sum()))
pd.concat({"전체 노선": report(main), "준분리 노선 제외": report(fit(morning, X_V2, drop_routes=separated))}, axis=0).round(4)

2025년 오전 승차가 모두 0인 노선: ['1100'] | 해당 쌍: 2


β  변화율 차이(%)  95% CI 하한(%)  95% CI 상한(%)       p
전체 노선     보행시간 2배당      0.3462    27.1165       13.9093       41.8550  0.0000
          IC 도로거리 1km당 -0.1236   -11.6237      -17.7447       -5.0472  0.0007
준분리 노선 제외 보행시간 2배당      0.3462    27.1165       13.9380       41.8193  0.0000
          IC 도로거리 1km당 -0.1236   -11.6237      -17.7312       -5.0629  0.0007

## 5. 다중공선성

In [6]:
X = sm.add_constant(pd.concat([morning[X_V2], pd.get_dummies(morning.노선, prefix="노선", drop_first=True, dtype=float)], axis=1))
print({c: round(variance_inflation_factor(X.values, list(X.columns).index(c)), 2) for c in X_V2})
print(f"셀 단위 상관 (log 보행시간, IC) = {data[['log_walk', 'ic_access']].corr().iloc[0, 1]:.2f}")

{'log_walk': np.float64(3.84), 'ic_access': np.float64(2.57)}
셀 단위 상관 (log 보행시간, IC) = 0.45


## 6. 보조 결과 — 보행시간 구간 더미 (해석용, 경계는 탐색적으로 설정)

In [7]:
aux = fit(morning, ["walk_20", "walk_20_60", "ic_access"])
ci = aux.conf_int()
pd.DataFrame({name: {"변화율 차이(%)": (np.exp(aux.params[col]) - 1) * 100,
                     "95% CI 하한(%)": (np.exp(ci.loc[col, 0]) - 1) * 100, "95% CI 상한(%)": (np.exp(ci.loc[col, 1]) - 1) * 100,
                     "p": aux.pvalues[col]}
              for name, col in [("보행 20분 이내 (vs 60분 초과)", "walk_20"), ("보행 20~60분 (vs 60분 초과)", "walk_20_60"),
                                ("IC 도로거리 1km당", "ic_access")]}).T.round(4)

,변화율 차이(%),95% CI 하한(%),95% CI 상한(%),p
보행 20분 이내 (vs 60분 초과),-44.4394,-60.6213,-21.6080,0.0008
보행 20~60분 (vs 60분 초과),-17.0331,-36.1926,7.8796,0.1634
IC 도로거리 1km당,-10.3297,-16.8419,-3.3074,0.0046


## 7. 종일 (강건성)

In [8]:
pd.concat({"오전(본)": report(main), "종일": report(fit(allday, X_V2))}, axis=0).round(4)

β  변화율 차이(%)  95% CI 하한(%)  95% CI 상한(%)       p
오전(본) 보행시간 2배당      0.3462    27.1165       13.9093       41.8550  0.0000
      IC 도로거리 1km당 -0.1236   -11.6237      -17.7447       -5.0472  0.0007
종일    보행시간 2배당      0.3548    27.8848        7.2026       52.5570  0.0063
      IC 도로거리 1km당 -0.0428    -4.1920       -9.1566        1.0440  0.1147

## 결론
결과 해석과 v1 대비 변화는 `../최종모형_v2.md`에 정리했다.